In [1]:
# =========================================================
# PS S6E4 - exp_20260410_042_xgb024_bias_refine
# Refine post-hoc bias tuning for 024 only
# no retraining, bias-only search on saved raw probabilities
# =========================================================

## Import / config

In [2]:
import os
import json
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score


class CFG:
    EXP_ID = "exp_20260410_042_xgb024_bias_refine"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    ID_COL = "id"
    TARGET = "Irrigation_Need"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    OOF_RAW_024 = NPY_PATH + "oof_xgb_digit_orderedte_min_proba.npy"
    PRED_RAW_024 = NPY_PATH + "pred_xgb_digit_orderedte_min_proba.npy"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

In [3]:
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))

def apply_bias(proba, bias):
    """
    bias: iterable of len 3
    apply in log-prob space, then renormalize
    """
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)

    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)

def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)

def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
y = train[CFG.TARGET].map(CFG.TARGET_MAP).values

oof_raw = np.load(CFG.OOF_RAW_024).astype(np.float32)
pred_raw = np.load(CFG.PRED_RAW_024).astype(np.float32)

raw_cv = balanced_accuracy_score(y, np.argmax(oof_raw, axis=1))
print(f"raw_cv = {raw_cv:.6f}")

raw_cv = 0.978635


In [4]:
# ---------------------------------------------------------
# Stage 1: coarse
# ---------------------------------------------------------
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw, y,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

stage1 best_bias: (np.float64(-1.0), np.float64(-1.0), np.float64(-0.30000000000000016)) score: 0.9796000070760241


In [5]:
# ---------------------------------------------------------
# Stage 2: local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw, y,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

stage2 best_bias: (np.float64(-1.04), np.float64(-1.04), np.float64(-0.40000000000000013)) score: 0.9796243298231895


In [6]:
# ---------------------------------------------------------
# Stage 3: ultra-local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw, y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

np.save(f"{CFG.OUTPUT_DIR}/oof_xgb_digit_orderedte_min_proba_biased_refined.npy", oof_biased)
np.save(f"{CFG.OUTPUT_DIR}/pred_xgb_digit_orderedte_min_proba_biased_refined.npy", pred_biased)
np.save(f"{CFG.OUTPUT_DIR}/best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

submission = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET: pd.Series(np.argmax(pred_biased, axis=1)).map(CFG.INV_TARGET_MAP)
})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_xgb_digit_orderedte_min_bias_refined.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260409_024_xgb_digit_orderedte_min",
    "raw_cv": float(raw_cv),
    "refined_biased_cv": float(final_cv),
    "best_bias": list(map(float, final_bias)),
    "search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
        "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
    }
}
save_json(cv_result, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("final_bias:", final_bias)
print("final_cv:", final_cv)
print("saved to:", CFG.OUTPUT_DIR)

stage3 best_bias: (np.float64(-1.0500000000000003), np.float64(-1.0500000000000003), np.float64(-0.42000000000000015)) score: 0.9796246655652068
final_bias: (np.float64(-1.0500000000000003), np.float64(-1.0500000000000003), np.float64(-0.42000000000000015))
final_cv: 0.9796246655652068
saved to: /kaggle/working/exp_20260410_042_xgb024_bias_refine
